# 01 · Data quality

Checks every one of the 9 raw Olist CSVs for missing values, duplicates, impossible dates, outliers, orphan rows and missing translations, before any analysis is trusted.

Output: `data_quality_report.md` in the project root, listing every issue, how many rows it affects and how it is handled.

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
pd.set_option("display.max_columns", 20)

FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "geolocation": "olist_geolocation_dataset.csv",
}
ZIP_COLUMNS = {c: str for c in ["customer_zip_code_prefix", "seller_zip_code_prefix", "geolocation_zip_code_prefix"]}

df = {name: pd.read_csv(RAW / f, dtype=ZIP_COLUMNS, encoding="utf-8-sig") for name, f in FILES.items()}

date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
for table, cols in date_cols.items():
    for c in cols:
        df[table][c] = pd.to_datetime(df[table][c])

pd.DataFrame({name: {"rows": len(d), "columns": d.shape[1]} for name, d in df.items()}).T

,rows,columns
orders,99441,8
order_items,112650,7
payments,103886,5
reviews,99224,7
customers,99441,5
sellers,3095,4
products,32951,9
category_translation,71,2
geolocation,1000163,5


In [2]:
# Every issue found below is recorded here and written to the report at the end.
issues = []

def log(table, issue, rows, handling):
    issues.append({"table": table, "issue": issue, "rows": int(rows), "handling": handling})
    print(f"{table:<22} {issue:<60} {rows:>8,}")

## 1. Missing values

Only columns with at least one missing value are shown.

In [3]:
nulls = []
for name, d in df.items():
    counts = d.isna().sum()
    for col, n in counts[counts > 0].items():
        nulls.append({"table": name, "column": col, "missing": n, "pct": round(100 * n / len(d), 2)})
nulls = pd.DataFrame(nulls).sort_values(["table", "missing"], ascending=[True, False])
nulls

,table,column,missing,pct
2,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
0,orders,order_approved_at,160,0.16
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01
10,products,product_length_cm,2,0.01
11,products,product_height_cm,2,0.01


Most of these gaps are expected, not errors:

- **Review comments** are optional, so most reviews have no title or message.
- **Delivery dates** are blank for orders that were cancelled or never shipped.
- **Product attributes** are blank for the same 610 products that have no category.

The one that matters for the analysis is delivered orders with no delivery date, checked next.

In [4]:
o = df["orders"]
delivered_no_date = o[(o.order_status == "delivered") & o.order_delivered_customer_date.isna()]
log("orders", "Delivered orders with no customer delivery date", len(delivered_no_date),
    "Excluded from delivery-time and late-rate metrics; kept in revenue.")
log("orders", "Orders with no approval time", o.order_approved_at.isna().sum(),
    "Not used in any metric; left as is.")
log("products", "Products with no category", df["products"].product_category_name.isna().sum(),
    "Labelled 'unknown' in category analysis.")
delivered_no_date[["order_id", "order_status", "order_purchase_timestamp", "order_delivered_carrier_date"]]

orders                 Delivered orders with no customer delivery date                     8
orders                 Orders with no approval time                                      160
products               Products with no category                                         610


,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-30 18:12:23
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-25 08:05:00
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-03 13:57:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-03 13:57:00
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-03 09:28:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-12 14:10:00
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-07-03 19:26:00


## 2. Duplicates

In [5]:
dup_rows = {name: int(d.duplicated().sum()) for name, d in df.items()}
dup_rows

{'orders': 0,
 'order_items': 0,
 'payments': 0,
 'reviews': 0,
 'customers': 0,
 'sellers': 0,
 'products': 0,
 'category_translation': 0,
 'geolocation': 261831}

In [6]:
keys = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "products": ["product_id"],
    "category_translation": ["product_category_name"],
}
{name: int(df[name].duplicated(subset=k).sum()) for name, k in keys.items()}

{'orders': 0,
 'order_items': 0,
 'payments': 0,
 'customers': 0,
 'sellers': 0,
 'products': 0,
 'category_translation': 0}

Every table's primary key is unique. The two exceptions are reviews and geolocation:

In [7]:
r = df["reviews"]
log("reviews", "review_id used on more than one order", r.review_id.duplicated().sum(),
    "Key is (review_id, order_id) instead of review_id alone.")
log("reviews", "Orders with more than one review", (r.groupby("order_id").size() > 1).sum(),
    "Scores averaged per order before any analysis.")
log("geolocation", "Exact duplicate rows", dup_rows["geolocation"],
    "Not used for analysis; customer and seller state come from their own tables.")
c = df["customers"]
log("customers", "customer_id values that belong to a repeat person",
    len(c) - c.customer_unique_id.nunique(),
    "All customer counts use customer_unique_id.")

reviews                review_id used on more than one order                             814
reviews                Orders with more than one review                                  547
geolocation            Exact duplicate rows                                          261,831
customers              customer_id values that belong to a repeat person               3,345


## 3. Impossible or suspicious dates

In [8]:
log("orders", "Delivered to customer before purchase",
    (o.order_delivered_customer_date < o.order_purchase_timestamp).sum(), "None found.")
log("orders", "Approved before purchase",
    (o.order_approved_at < o.order_purchase_timestamp).sum(), "None found.")
log("orders", "Handed to carrier before purchase",
    (o.order_delivered_carrier_date < o.order_purchase_timestamp).sum(),
    "Carrier date treated as unreliable; not used for headline metrics.")
log("orders", "Delivered to customer before handed to carrier",
    (o.order_delivered_customer_date < o.order_delivered_carrier_date).sum(),
    "Carrier date treated as unreliable; customer delivery date is used.")

orders                 Delivered to customer before purchase                               0
orders                 Approved before purchase                                            0
orders                 Handed to carrier before purchase                                 166
orders                 Delivered to customer before handed to carrier                     23


In [9]:
monthly = o.groupby(o.order_purchase_timestamp.dt.to_period("M")).size()
log("orders", "Orders in sparse months (2016, Sep-Oct 2018)",
    monthly[(monthly.index < "2017-01") | (monthly.index > "2018-08")].sum(),
    "Analysis limited to Jan 2017 - Aug 2018 (20 full months).")
monthly.to_frame("orders").T

orders                 Orders in sparse months (2016, Sep-Oct 2018)                      349


order_purchase_timestamp,2016-09,2016-10,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,...,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10
orders,4,324,1,800,1780,2682,2404,3700,3245,4026,...,7269,6728,7211,6939,6873,6167,6292,6512,16,4


## 4. Outliers and impossible values

In [10]:
i = df["order_items"]
log("order_items", "Freight costs more than the item", (i.freight_value > i.price).sum(),
    "Real, not an error: cheap items shipped far. Kept; analysed in query 11.")
log("order_items", "Price of zero or less", (i.price <= 0).sum(), "None found.")
log("order_items", "Freight of zero", (i.freight_value == 0).sum(),
    "Free-shipping promotions. Kept.")
log("reviews", "Review score outside 1-5", (~r.review_score.between(1, 5)).sum(), "None found.")
p = df["products"]
log("products", "Products with weight of 0 g", (p.product_weight_g == 0).sum(),
    "Product weight is not used in the analysis. Kept.")
i[["price", "freight_value"]].describe(percentiles=[0.5, 0.99]).round(2)

order_items            Freight costs more than the item                                4,124
order_items            Price of zero or less                                               0
order_items            Freight of zero                                                   383
reviews                Review score outside 1-5                                            0
products               Products with weight of 0 g                                         4


,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
50%,74.99,16.26
99%,890.00,84.52
max,6735.00,409.68


In [11]:
pay = df["payments"]
log("payments", "Payment type 'not_defined'", (pay.payment_type == "not_defined").sum(),
    "Excluded from payment-mix analysis.")
log("payments", "Payments with 0 instalments", (pay.payment_installments == 0).sum(),
    "Treated as a single payment.")
log("payments", "Payment value of zero", (pay.payment_value == 0).sum(),
    "Vouchers covering the full amount. Kept.")

payments               Payment type 'not_defined'                                          3
payments               Payments with 0 instalments                                         2
payments               Payment value of zero                                               9


## 5. Orphan rows

Rows that point to a record that does not exist in the parent table.

In [12]:
log("order_items", "Items whose product is missing", (~i.product_id.isin(p.product_id)).sum(), "None found.")
log("order_items", "Items whose seller is missing", (~i.seller_id.isin(df["sellers"].seller_id)).sum(), "None found.")
log("orders", "Orders whose customer is missing", (~o.customer_id.isin(c.customer_id)).sum(), "None found.")
log("orders", "Orders with no items", (~o.order_id.isin(i.order_id)).sum(),
    "Almost all cancelled or unavailable; they drop out of revenue via the inner join.")
log("orders", "Orders with no payment", (~o.order_id.isin(pay.order_id)).sum(), "Ignored (1 order).")
o[~o.order_id.isin(i.order_id)].order_status.value_counts()

order_items            Items whose product is missing                                      0
order_items            Items whose seller is missing                                       0
orders                 Orders whose customer is missing                                    0
orders                 Orders with no items                                              775
orders                 Orders with no payment                                              1


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

## 6. Missing English category names

In [13]:
t = df["category_translation"]
untranslated = p[p.product_category_name.notna() & ~p.product_category_name.isin(t.product_category_name)]
log("products", "Products whose category has no English name", len(untranslated),
    "Portuguese name used instead (pc_gamer, portateis_cozinha_e_preparadores_de_alimentos).")
untranslated.product_category_name.value_counts()

products               Products whose category has no English name                        13


product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

## 7. Write the report

In [14]:
report = pd.DataFrame(issues)
found = report[report.rows > 0]
clean = report[report.rows == 0]

lines = [
    "# Data quality report",
    "",
    "Generated by `notebooks/01_data_quality.ipynb` from the 9 raw Olist CSVs.",
    "",
    f"{len(report)} checks run: {len(found)} found issues, {len(clean)} passed.",
    "",
    "## Issues found",
    "",
    "| Table | Issue | Rows | How it is handled |",
    "|---|---|---:|---|",
]
lines += [f"| `{r.table}` | {r.issue} | {r.rows:,} | {r.handling} |" for r in found.itertuples()]
lines += ["", "## Checks that passed", ""]
lines += [f"- `{r.table}`: {r.issue}" for r in clean.itertuples()]

Path("../data_quality_report.md").write_text("\n".join(lines) + "\n")
found

,table,issue,rows,handling
0,orders,Delivered orders with no customer delivery date,8,Excluded from delivery-time and late-rate metr...
1,orders,Orders with no approval time,160,Not used in any metric; left as is.
2,products,Products with no category,610,Labelled 'unknown' in category analysis.
3,reviews,review_id used on more than one order,814,"Key is (review_id, order_id) instead of review..."
4,reviews,Orders with more than one review,547,Scores averaged per order before any analysis.
5,geolocation,Exact duplicate rows,261831,Not used for analysis; customer and seller sta...
6,customers,customer_id values that belong to a repeat person,3345,All customer counts use customer_unique_id.
9,orders,Handed to carrier before purchase,166,Carrier date treated as unreliable; not used f...
10,orders,Delivered to customer before handed to carrier,23,Carrier date treated as unreliable; customer d...
11,orders,"Orders in sparse months (2016, Sep-Oct 2018)",349,Analysis limited to Jan 2017 - Aug 2018 (20 fu...
